[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decimal-labs/decimalai-python/blob/main/examples/version-aware-loop/manifest_change.ipynb)

# Version-Aware Loop: Manifest Changes & Impact Reports

**DecimalAI's core differentiator: automatic detection of agent changes and their impact on your training data.**

This notebook dives deeper than the quickstart into:
- How manifests are auto-detected from tools, models, and prompts
- What triggers a new manifest version
- How the impact report classifies every trace as **keep / repair / replay / drop**
- When to use mechanical repair vs. LLM replay

> **No LLM API key required.** All examples use mock functions.

## What Is a Manifest?

A **manifest** is a snapshot of your agent's configuration at a point in time:
- **Tools**: names, schemas, descriptions
- **Models**: provider, model name, temperature
- **Prompts**: system prompts, templates
- **Subagents**: names and configurations

DecimalAI hashes this snapshot. When the hash changes, a new manifest version is created
and all existing traces are classified against the new version.

## Setup

In [ ]:
!pip install -q decimalai

In [ ]:
import os
os.environ["DECIMAL_API_KEY"] = "dai_sk_..."  # ← Replace with your key

import decimalai
decimalai.init()

## Scenario 1: Tool Rename

The most common change. You rename a tool for clarity but the
functionality stays the same. DecimalAI classifies affected traces
as **repair** — they can be mechanically fixed without re-running.

In [ ]:
# ── v1: Original agent ──

@decimalai.trace(agent_name="demo-agent")
def agent_v1(query: str) -> str:
    decimalai.log_tool_call(name="get_stock_price", input={"ticker": "AAPL"}, output={"price": 178.52})
    return f"AAPL is at $178.52"

# Generate 3 traces with v1
for q in ["What is AAPL price?", "Check TSLA stock", "NVDA current price"]:
    agent_v1(q)

print("✅ 3 traces sent with tool: get_stock_price")

In [ ]:
# ── v2: Renamed tool ──

@decimalai.trace(agent_name="demo-agent")
def agent_v2(query: str) -> str:
    # Same function, better name
    decimalai.log_tool_call(name="lookup_ticker", input={"ticker": "AAPL"}, output={"price": 178.52})
    return f"AAPL is at $178.52"

agent_v2("What is AAPL price?")

print("✅ Manifest v2 auto-detected!")
print("   Changed: get_stock_price → lookup_ticker")
print("   Impact: 3 traces classified as 'repair'")

### What Happened

```
Manifest v1: tools=[get_stock_price]     → 3 traces
Manifest v2: tools=[lookup_ticker]        → 1 trace

Impact Report:
  keep:   0  (no traces are unaffected — all used the renamed tool)
  repair: 3  (tool name changed, traces can be mechanically updated)
  replay: 0  (no schema changes requiring re-execution)
  drop:   0  (no tools removed entirely)
```

**Repair** means DecimalAI can automatically rewrite `get_stock_price` → `lookup_ticker`
in the trace data. No LLM cost, instant.

## Scenario 2: Tool Added

Adding a new tool doesn't break existing traces — they just didn't use it.
All existing traces are classified as **keep**.

In [ ]:
# ── v3: Added a new tool ──

@decimalai.trace(agent_name="demo-agent")
def agent_v3(query: str) -> str:
    decimalai.log_tool_call(name="lookup_ticker", input={"ticker": "AAPL"}, output={"price": 178.52})
    # NEW: Also log the new tool
    decimalai.log_tool_call(name="get_earnings", input={"ticker": "AAPL"}, output={"eps": 6.42})
    return f"AAPL: $178.52, EPS: $6.42"

agent_v3("AAPL price and earnings")

print("✅ Manifest v3: added get_earnings")
print("   Impact: existing traces → keep (new tool doesn't invalidate old data)")

## Scenario 3: Tool Removed

Removing a tool that existing traces used is the most disruptive change.
Traces that called the removed tool are classified as **drop** — they
reference a capability that no longer exists and can't be mechanically fixed.

In [ ]:
# ── v4: Removed lookup_ticker, kept only get_earnings ──

@decimalai.trace(agent_name="demo-agent")
def agent_v4(query: str) -> str:
    # Only uses get_earnings now
    decimalai.log_tool_call(name="get_earnings", input={"ticker": "AAPL"}, output={"eps": 6.42})
    return f"AAPL EPS: $6.42"

agent_v4("AAPL earnings")

print("✅ Manifest v4: removed lookup_ticker")
print("   Impact:")
print("     - Traces using lookup_ticker → drop")
print("     - Traces using only get_earnings → keep")

## Scenario 4: Schema Change (Replay)

When a tool's **schema** changes (new required parameter, different output format),
traces can't be mechanically fixed. They need to be **replayed** — re-executed
through the updated agent so the LLM learns the new schema.

In [ ]:
# ── v5: Changed tool schema ──

@decimalai.trace(agent_name="demo-agent")
def agent_v5(query: str) -> str:
    # get_earnings now requires a 'period' parameter
    decimalai.log_tool_call(
        name="get_earnings",
        input={"ticker": "AAPL", "period": "quarterly"},  # NEW: period param
        output={"eps": 1.52, "period": "Q4 2025"},  # Richer output
    )
    return f"AAPL Q4 2025 EPS: $1.52"

agent_v5("AAPL quarterly earnings")

print("✅ Manifest v5: get_earnings schema changed")
print("   Impact: old traces using get_earnings → replay")
print("   (schema changed, mechanical repair isn't sufficient)")

## Using the SDK to Query Impact

You can also use the `register_manifest` API to explicitly declare
your agent's configuration and see the impact report programmatically.

In [ ]:
# Explicit manifest registration
result = decimalai.register_manifest(
    agent_name="demo-agent",
    tools=[
        {"name": "get_earnings", "schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string"},
                "period": {"type": "string", "enum": ["quarterly", "annual"]},
            },
            "required": ["ticker", "period"],
        }},
    ],
    models={"default": {"provider": "openai", "model": "gpt-4o"}},
    prompts={"system": "You are a financial analyst assistant."},
)

print(f"Manifest ID: {result.get('manifest_id', 'N/A')}")
print(f"Compatibility: {result.get('compatibility', {})}")
print(f"Impact: {result.get('impact_summary', {})}")
print()
print("📊 Open dashboard: https://app.decimal.ai/agents/demo-agent")

## Summary: The Four Classifications

| Classification | Trigger | Fix | Cost |
|---------------|---------|-----|------|
| **Keep** | Trace doesn't use any changed component | None needed | Free |
| **Repair** | Tool renamed, prompt template changed | Mechanical string replacement | Free |
| **Replay** | Schema changed, model swapped | Re-run through updated agent | LLM cost |
| **Drop** | Tool removed entirely | Cannot fix — exclude from dataset | Free |

### Why This Matters for Training

If you build a fine-tuning dataset from traces without checking compatibility:
- **Stale tool calls** teach the model to call non-existent tools
- **Wrong schemas** teach the model to pass outdated parameters
- **Mixed versions** create inconsistent training signal

DecimalAI catches all of this automatically.

## Next Steps

- 📖 [Quickstart](../quickstart/quickstart.ipynb) — Start from the basics
- 📖 [Evaluations](../evaluations/builtin_evaluators.ipynb) — Score your traces
- 📖 [Build Datasets](../datasets-and-training/build_sft_dataset.ipynb) — Create clean training data